# Scalable indicator calculations with Dask

This tutorial demonstrates how to run indicator calculations on larger datasets using **Dask** for lazy evaluation and parallel computation.

> **Key Concept**: Before continuing with this hands-on tutorial, we strongly recommend reading the concept page on [Scalability and performance](../concepts/scalability_performance.rst), which explains Dask chunking principles, spatial vs. temporal chunking, task vs. peer-to-peer rechunking engines, and worker memory management.

Datasets loaded via **earthkit-data** (e.g. from GRIB or NetCDF files) can be passed directly to indicators or converted to lazy chunked arrays.


In [1]:
import earthkit.data as ekd

import earthkit.climate as ekc

## 1. Loading a chunked lazy dataset

When working with large climate datasets (e.g. ERA5 or CMIP6), data arrays are loaded as **dask-backed DataArrays**.
Here we load a 3D spatial-temporal daily maximum temperature dataset (`tasmax`) backed by Dask arrays from `earthkit-climate-sample`.

You can verify that a DataArray is dask-backed by checking `.chunks` or inspecting the underlying data array type with `isinstance(tasmax.data, da.Array)`.


In [2]:
import dask.array as da

tasmax = ekd.from_source("earthkit-climate-sample", "synthetic-daily-dask-temperature").to_xarray()

# Check Dask chunk structure and underlying array type
print("Is dask-backed:", isinstance(tasmax.data, da.Array))
print("Chunk sizes (time, lat, lon):", tasmax.chunks)
tasmax

Is dask-backed: True
Chunk sizes (time, lat, lon): ((365, 365, 365, 365, 365, 365, 365, 365, 365, 365, 2), (10, 10), (10, 10))


/home/cuadradot/predictia_projects/git/c3s-indices/earthkit-climate/src/earthkit/climate/sample_source.py:185: UserWarning: earthkit-climate-sample datasets are made available for demonstration purposes only. Files are not guaranteed to be available long-term and may change over time. Please use official channels to obtain the contained datasets reliably for other purposes.
  warnings.warn(


<xarray.DataArray 'tasmax' (time: 3652, lat: 20, lon: 20)> Size: 12MB
dask.array<normal, shape=(3652, 20, 20), dtype=float64, chunksize=(365, 10, 10), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[us] 29kB 2010-01-01 2010-01-02 ... 2019-12-31
  * lat      (lat) float64 160B 35.0 36.32 37.63 38.95 ... 57.37 58.68 60.0
  * lon      (lon) float64 160B -10.0 -7.895 -5.789 -3.684 ... 25.79 27.89 30.0
Attributes:
    units:          K
    standard_name:  air_temperature
    cell_methods:   time: maximum

## 2. Computing indicators lazily

When passing a dask-backed DataArray to **earthkit-climate** indicator functions, the index calculation is evaluated **lazily**.
The function returns a new DataArray containing a Dask computational task graph without performing heavy numerical computation immediately.


In [3]:
%%time
# Construct lazy task graph (executes almost instantaneously)
hot_days_lazy = ekc.indicators.tx_days_above(tasmax, thresh="300 K", freq="YS")
hot_days_lazy

CPU times: user 55.4 ms, sys: 3.92 ms, total: 59.3 ms
Wall time: 58.9 ms


<xarray.DataArray 'tx_days_above' (time: 10, lat: 20, lon: 20)> Size: 32kB
dask.array<where, shape=(10, 20, 20), dtype=float64, chunksize=(1, 10, 10), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[us] 80B 2010-01-01 2011-01-01 ... 2019-01-01
  * lat      (lat) float64 160B 35.0 36.32 37.63 38.95 ... 57.37 58.68 60.0
  * lon      (lon) float64 160B -10.0 -7.895 -5.789 -3.684 ... 25.79 27.89 30.0
Attributes:
    units:          days
    standard_name:  number_of_days_with_air_temperature_above_threshold
    cell_methods:   time: maximum time: sum over days
    history:        [2026-09-23 10:26:10] tx_days_above: TX_DAYS_ABOVE(tasmax...
    long_name:      The number of days with maximum temperature above 300 k
    description:    Annual number of days where daily maximum temperature exc...

## 3. Triggering computation and saving results

To execute the task graph across available CPU threads, call `.compute()` or write directly to a NetCDF/Zarr store using `to_netcdf` or `to_zarr`.


In [4]:
%%time
# Compute the task graph (triggers numerical calculation)
hot_days_computed = hot_days_lazy.compute()
hot_days_computed

CPU times: user 187 ms, sys: 25.5 ms, total: 212 ms
Wall time: 173 ms


<xarray.DataArray 'tx_days_above' (time: 10, lat: 20, lon: 20)> Size: 32kB
array([[[23., 20., 29., ..., 27., 20., 37.],
        [26., 32., 22., ..., 33., 41., 21.],
        [18., 24., 22., ..., 28., 28., 26.],
        ...,
        [24., 27., 17., ..., 32., 30., 28.],
        [21., 24., 31., ..., 33., 28., 29.],
        [20., 29., 23., ..., 19., 25., 33.]],

       [[23., 29., 27., ..., 18., 19., 18.],
        [22., 22., 25., ..., 26., 22., 25.],
        [29., 26., 19., ..., 19., 22., 19.],
        ...,
        [23., 28., 25., ..., 24., 22., 21.],
        [24., 26., 26., ..., 29., 29., 19.],
        [35., 30., 21., ..., 12., 29., 21.]],

       [[17., 37., 21., ..., 21., 15., 26.],
        [14., 20., 22., ..., 31., 22., 29.],
        [18., 19., 28., ..., 26., 25., 30.],
        ...,
...
        ...,
        [18., 22., 34., ..., 27., 31., 25.],
        [23., 22., 23., ..., 23., 32., 34.],
        [29., 29., 22., ..., 24., 23., 20.]],

       [[29., 21., 32., ..., 22., 22., 20.],
        [24., 28., 31., ..., 24., 26., 30.],
        [21., 25., 28., ..., 18., 22., 28.],
        ...,
        [27., 23., 23., ..., 25., 17., 19.],
        [23., 35., 21., ..., 21., 21., 32.],
        [25., 27., 29., ..., 26., 18., 19.]],

       [[24., 20., 24., ..., 26., 28., 25.],
        [31., 19., 15., ..., 26., 29., 23.],
        [25., 28., 24., ..., 18., 25., 16.],
        ...,
        [16., 26., 25., ..., 20., 30., 37.],
        [27., 25., 19., ..., 18., 21., 16.],
        [24., 18., 20., ..., 23., 24., 30.]]], shape=(10, 20, 20))
Coordinates:
  * time     (time) datetime64[us] 80B 2010-01-01 2011-01-01 ... 2019-01-01
  * lat      (lat) float64 160B 35.0 36.32 37.63 38.95 ... 57.37 58.68 60.0
  * lon      (lon) float64 160B -10.0 -7.895 -5.789 -3.684 ... 25.79 27.89 30.0
Attributes:
    units:          days
    standard_name:  number_of_days_with_air_temperature_above_threshold
    cell_methods:   time: maximum time: sum over days
    history:        [2026-09-23 10:26:10] tx_days_above: TX_DAYS_ABOVE(tasmax...
    long_name:      The number of days with maximum temperature above 300 k
    description:    Annual number of days where daily maximum temperature exc...

## 4. Groupby considerations for daily climatologies and percentiles

Operations involving daily climatologies or percentile thresholds require contiguous time series for each spatial point.

* **Best practice**: Ensure the time dimension is unchunked (or chunked in multi-year blocks) before computing daily percentiles or climatologies.
* **Rechunking**: If your disk data is time-sliced (e.g. 1 day per chunk), rechunk spatially: `tasmax.chunk({"time": -1, "lat": 5, "lon": 5})`.


In [5]:
tasmax_rechunked = tasmax.chunk({"time": -1, "lat": 5, "lon": 5})
per90_lazy = ekc.utils.climatology.rolling_percentiles(tasmax_rechunked, p=90, window_width=5)
per90_lazy

/home/cuadradot/predictia_projects/git/c3s-indices/earthkit-climate/.venv/lib/python3.11/site-packages/dask/array/core.py:5201: PerformanceWarning: Increasing number of chunks by factor of 12
  result = blockwise(


<xarray.DataArray 'tasmax' (lat: 20, lon: 20, dayofyear: 366, percentile: 1)> Size: 1MB
dask.array<transpose, shape=(20, 20, 366, 1), dtype=float64, chunksize=(2, 2, 366, 1), chunktype=numpy.ndarray>
Coordinates:
  * lat         (lat) float64 160B 35.0 36.32 37.63 38.95 ... 57.37 58.68 60.0
  * lon         (lon) float64 160B -10.0 -7.895 -5.789 ... 25.79 27.89 30.0
  * dayofyear   (dayofyear) int64 3kB 1 2 3 4 5 6 7 ... 361 362 363 364 365 366
  * percentile  (percentile) int64 8B 90
Attributes:
    units:               K
    standard_name:       air_temperature
    cell_methods:        time: maximum
    climatology_bounds:  ['2010-01-01', '2019-12-31']
    window:              5
    alpha:               0.3333333333333333
    beta:                0.3333333333333333
    history:             [2026-09-23 10:26:11] per: percentile_doy(arr=tasmax...